# GCM BA - 4. Verification of F1

On the EC-Earth3 grid, over the **LISFLOOD domain** (mask `ecearth3` of notebook 00c, no buffer; F1 itself is computed
on the whole globe), with the statistics of every dataset (cached):

1. **integrity** of the adjusted runs;
2. **calibration**: EC-Earth3 historical + F1 against ERA5 and against raw EC-Earth3 (bias maps, scores, distributions,
   dry days, extremes). Same period as the training: QDM maps each quantile, so the adjusted historical run should match
   ERA5 closely (in-sample check). EC-Earth3 and ERA5 are not synchronised day by day: against ERA5 only the RMSE
   components that need no pairing are given (bias, spatial variability); RMSE and correlation for adjusted vs raw;
3. **signals preserved**: climate change (2025 / 2050 original - historical) and hosing (hosing - original), raw vs
   adjusted: maps, pixel-by-pixel regression (slope 1 = preserved), annual cycle, extremes, dry days, quantiles.

Figures and tables: `outputs/` next to this notebook (not in git); statistics cache on EOS.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, json, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS_CORE = cfg["variables"]                                   # pr, tasmax, tasmin: required
VARS_ALL = VARS_CORE                # rsds not bias-adjusted (to add it back: VARS_CORE + cfg.get("extra_variables", []))
GCM_RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]

CAL_PERIOD = ("1981", "2014")      # years of ERA5 / EC-Earth3 historical used for F1 (EC-Earth3 historical ends in 2014)
RSDS_FIRST_YEAR = "2005"           # ERA5 radiation starts in 2005: shorter period for rsds

def period_of(v):
    """Calibration period of a variable: CAL_PERIOD, starting in 2005 for rsds."""
    return (max(CAL_PERIOD[0], RSDS_FIRST_YEAR), CAL_PERIOD[1]) if v == "rsds" else CAL_PERIOD

F1_TAG = f"F1_{CAL_PERIOD[0]}-{CAL_PERIOD[1]}"
import validation_utils as vu
OVERWRITE_CACHE = False
DOMAIN = vu.init(cfg, "domain", borders_res="01M")       # LISFLOOD domain (ecearth3 mask): statistics and maps over land
OUTF1 = bu.gcm_ba_dir(cfg, "GCM_bias_adjusted_0.7deg")    # bias-adjusted GCM runs (notebook 3)
CACHE = bu.gcm_ba_dir(cfg, "4_verification", "cache")
OUT = Path.cwd() / "outputs"; OUT.mkdir(exist_ok=True)
LABEL = {"gcm_historical": "historical", "gcm_2025_nohosing": "2025 original", "gcm_2025_hosing": "2025 hosing",
         "gcm_2050_nohosing": "2050 original", "gcm_2050_hosing": "2050 hosing"}
PAIRS = [("gcm_2025_nohosing", "gcm_historical", "2025 original - historical"),
         ("gcm_2050_nohosing", "gcm_historical", "2050 original - historical"),
         ("gcm_2025_hosing", "gcm_2025_nohosing", "2025 hosing - original"),
         ("gcm_2050_hosing", "gcm_2050_nohosing", "2050 hosing - original")]
print("adjusted runs:", OUTF1, "| figures:", OUT)

## 0. Statistics (computed once, cached)

In [ ]:
def inputs(name):
    """Raw inputs (EC-Earth3 grid), standardised when read; ERA5 (CDO) and historical over the calibration period."""
    cal = name in ("era5", "gcm_historical")
    return {v: bu.gcm_ba_input(cfg, name, v, period_of(v) if cal else None) for v in VARS_ALL if bu.gcm_ba_available(cfg, name, v)}

def adjusted(name):
    f = OUTF1 / f"{name}_{F1_TAG}.nc"
    if not f.exists():
        return {}
    ds = xr.open_dataset(f, chunks={"time": 31}, **bu._TIME_KW)
    return {v: ds[v] for v in VARS_ALL if v in ds}

S_era5 = vu.compute_stats(inputs("era5"), CACHE / "era5_stats.nc", OVERWRITE_CACHE, label="ERA5 (EC-Earth3 grid)")
RAW, ADJ = {}, {}
for name, lab in LABEL.items():
    a = adjusted(name)
    if not a:
        print(f"[missing] {lab}: run notebook 3"); continue
    RAW[name] = vu.compute_stats(inputs(name), CACHE / f"{name}_raw_stats.nc", OVERWRITE_CACHE, label=f"{lab} raw")
    ADJ[name] = vu.compute_stats(a, CACHE / f"{name}_{F1_TAG}_stats.nc", OVERWRITE_CACHE, label=f"{lab} + F1")
VPLOT = [v for v in ("pr", "tasmax", "tasmin", "dtr", "rsds") if vu.has(S_era5, v) or any(vu.has(s, v) for s in ADJ.values())]
PAIRS = [p for p in PAIRS if p[0] in ADJ and p[1] in ADJ]
print("runs:", [LABEL[n] for n in ADJ], "| variables:", VPLOT)

## 1. Integrity of the adjusted runs

In [ ]:
rows = []
for name, s in ADJ.items():
    for v in [x for x in VARS_ALL if f"period_{x}" in s.attrs]:
        rows.append({"run": LABEL[name], "var": v, "period": s.attrs[f"period_{v}"], "n days": s.attrs[f"ndays_{v}"],
                     "min": round(s.attrs[f"min_{v}"], 3), "max": round(s.attrs[f"max_{v}"], 3)})
    rows.append({"run": LABEL[name], "var": "tasmin > tasmax (frac)", "period": "", "n days": "",
                 "min": s.attrs.get("frac_tasmin_gt_tasmax"), "max": ""})
display(pd.DataFrame(rows).set_index(["run", "var"]))

## 2. Calibration: EC-Earth3 historical + F1 vs ERA5 (in-sample)

### 2.1 Bias against ERA5: raw vs adjusted (pr in %; dots: |Welch t| > 2 on the yearly values)

In [ ]:
H_raw, H_adj = RAW.get("gcm_historical"), ADJ.get("gcm_historical")
SETS = {"ERA5": S_era5, "raw": H_raw, "F1": H_adj}
VCAL = [v for v in VPLOT if all(vu.has(s, v) for s in SETS.values())]
for v in VCAL:
    fig, axes = vu.map_axes(2, 3)
    lim = None
    for r, lab in enumerate(("raw", "F1")):
        sig = vu.welch_sig(vu.yearly(SETS[lab], v), vu.yearly(S_era5, v))
        for c, s in enumerate(("ANN", "DJF", "JJA")):
            d = vu.diff(vu.seas(SETS[lab], v, s), vu.seas(S_era5, v, s), v)
            if lim is None:
                lim = float(np.nanpercentile(np.abs(d.where(vu.mask_for(d))), 98)) or 1.0
            vu.draw(axes[r, c], d, f"historical {lab} - ERA5: {v} {s}", cmap=vu.DCMAP[v], sym=True, lim=lim,
                    sig=sig if s == "ANN" else None, label=vu.dlabel(v))
    plt.tight_layout(); plt.savefig(OUT / f"calibration_bias_{v}.png", dpi=110); plt.show()

### 2.2 Scores (climatologies, over the domain)

In [ ]:
rows_e, rows_r = [], []
for v in VCAL:
    for s in ("ANN", "DJF", "MAM", "JJA", "SON"):
        ref, raw_, adj_ = vu.seas(S_era5, v, s), vu.seas(H_raw, v, s), vu.seas(H_adj, v, s)
        for lab, x in (("raw", raw_), ("F1", adj_)):
            rows_e.append({"var": v, "season": s, "dataset": lab, **{k: round(val, 3) for k, val in vu.scores_no_pairing(x, ref).items()}})
        sc = vu.scores(adj_, raw_)
        rows_r.append({"var": v, "season": s, **{k: round(sc[k], 3) for k in ("bias", "RMSE", "corr", "sd ratio")}})
t1 = pd.DataFrame(rows_e).pivot_table(index=["var", "season"], columns="dataset", values=["bias", "sd ratio", "sd diff"])
t1.to_csv(OUT / "calibration_scores_vs_era5.csv"); print("Against ERA5 (bias, spatial variability):"); display(t1)
t2 = pd.DataFrame(rows_r).set_index(["var", "season"])
t2.to_csv(OUT / "calibration_scores_adjusted_vs_raw.csv"); print("Adjusted against raw:"); display(t2)

### 2.3 Distributions (Q-Q against ERA5), dry days, extremes

In [ ]:
keys = [k for k in ("pr_q_wet", "tasmax_q", "tasmin_q", "dtr_q", "rsds_q") if all(k in s for s in SETS.values())]
fig, axes = plt.subplots(1, len(keys), figsize=(4.5 * len(keys), 4.2), squeeze=False)
for ax, k in zip(axes[0], keys):
    ref = S_era5[k].values
    for lab, col in (("raw", "k"), ("F1", "tab:red")):
        ax.plot(ref, SETS[lab][k].values, "o-", ms=3, color=col, label=lab)
    lo, hi = np.nanmin(ref), np.nanmax(ref)
    ax.plot([lo, hi], [lo, hi], color="grey", lw=0.8, ls="--")
    ax.set_xlabel("ERA5 quantile"); ax.set_ylabel("EC-Earth3 historical quantile")
    ax.set_title(k.replace("_q_wet", " (wet days)").replace("_q", ""))
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "calibration_qq.png", dpi=110); plt.show()
print("pooled dry-day fraction:", {lab: round(s.attrs.get("pooled_dry_frac", np.nan), 3) for lab, s in SETS.items()})

rows = []
for kind, name in (("dry", "dry-day fraction"), ("sdii", "wet-day intensity")):
    if all(f"pr_year_{kind}" in s for s in SETS.values()):
        for lab in ("raw", "F1"):
            rows.append({"index": name, "comparison": f"{lab} vs ERA5", **{k: round(x, 3) for k, x in vu.scores_no_pairing(vu.seas(SETS[lab], "pr", "ANN", kind), vu.seas(S_era5, "pr", "ANN", kind)).items()}})
for v, (how, name) in vu.EXT.items():
    if all(f"{v}_year_ext" in s for s in SETS.values()):
        for lab in ("raw", "F1"):
            rows.append({"index": name, "comparison": f"{lab} vs ERA5", **{k: round(x, 3) for k, x in vu.scores_no_pairing(vu.seas(SETS[lab], v, "ANN", "ext"), vu.seas(S_era5, v, "ANN", "ext")).items()}})
display(pd.DataFrame(rows).set_index(["index", "comparison"]))

## 3. Signals preserved? raw vs adjusted deltas (same grid)

### 3.1 Delta maps (pr in %): raw, adjusted, difference

In [ ]:
def delta(a, b, v, season="ANN", kind="mean"):
    return vu.diff(vu.seas(a, v, season, kind), vu.seas(b, v, season, kind), v)

for a, b, lab in PAIRS:
    vs = [v for v in VPLOT if all(vu.has(s, v) for s in (RAW[a], RAW[b], ADJ[a], ADJ[b]))]
    fig, axes = vu.map_axes(len(vs), 3)
    for r, v in enumerate(vs):
        d_raw, d_adj = delta(RAW[a], RAW[b], v), delta(ADJ[a], ADJ[b], v)
        lim = float(np.nanpercentile(np.abs(d_raw.where(vu.mask_for(d_raw))), 98)) or 1.0
        vu.draw(axes[r, 0], d_raw, f"raw: {v}", cmap=vu.DCMAP[v], sym=True, lim=lim, label=vu.dlabel(v),
                sig=vu.welch_sig(vu.yearly(RAW[a], v), vu.yearly(RAW[b], v)))
        vu.draw(axes[r, 1], d_adj, f"F1: {v}", cmap=vu.DCMAP[v], sym=True, lim=lim, label=vu.dlabel(v),
                sig=vu.welch_sig(vu.yearly(ADJ[a], v), vu.yearly(ADJ[b], v)))
        vu.draw(axes[r, 2], d_adj - d_raw, f"F1 - raw delta: {v}", cmap="PiYG", sym=True, lim=lim, label=vu.dlabel(v))
    fig.suptitle(lab, y=1.0)
    plt.tight_layout(); plt.savefig(OUT / f"delta_{a}_minus_{b}.png", dpi=110, bbox_inches="tight"); plt.show()

### 3.2 Is the delta preserved? Regression of the adjusted delta on the raw delta, pixel by pixel

Preserved: slope ~ 1, corr ~ 1, mean ratio ~ 1. QDM preserves the change quantile by quantile by construction
(exactly for additive variables); the fixed WDF may modify the dry-day part of the precipitation change.

In [ ]:
rows = []
INDICES = [(v, "mean", v) for v in VPLOT] + [(v, "ext", vu.EXT[v][1]) for v in vu.EXT] + \
          [("pr", "dry", "dry-day fraction"), ("pr", "sdii", "wet-day intensity")]
for a, b, lab in PAIRS:
    for v, kind, name in INDICES:
        key = {"mean": f"{v}_year_mean", "ext": f"{v}_year_ext", "dry": "pr_year_dry", "sdii": "pr_year_sdii"}[kind]
        if not all(key in s for s in (RAW[a], RAW[b], ADJ[a], ADJ[b])):
            continue
        for season in (("ANN", "DJF", "JJA") if kind == "mean" else ("ANN",)):
            if kind == "dry":
                d_raw = 100 * (vu.seas(RAW[a], v, season, kind) - vu.seas(RAW[b], v, season, kind))
                d_adj = 100 * (vu.seas(ADJ[a], v, season, kind) - vu.seas(ADJ[b], v, season, kind))
            else:
                d_raw, d_adj = delta(RAW[a], RAW[b], v, season, kind), delta(ADJ[a], ADJ[b], v, season, kind)
            rows.append({"pair": lab, "index": name, "season": season, **{k: round(x, 3) for k, x in vu.regression(d_raw, d_adj).items()}})
pres = pd.DataFrame(rows).set_index(["pair", "index", "season"])
pres.to_csv(OUT / "delta_preservation.csv"); display(pres)

### 3.3 Annual cycle of the deltas over the domain, and deltas quantile by quantile

In [ ]:
fig, axes = plt.subplots(len(PAIRS), len(VPLOT), figsize=(4.4 * len(VPLOT), 3.1 * len(PAIRS)), squeeze=False)
for r, (a, b, lab) in enumerate(PAIRS):
    for c, v in enumerate(VPLOT):
        ax = axes[r, c]
        if not all(vu.has(s, v, "clim_month") for s in (RAW[a], RAW[b], ADJ[a], ADJ[b])):
            ax.set_visible(False); continue
        for src, col, nm in ((RAW, "k", "raw"), (ADJ, "tab:red", "F1")):
            ma, mb = vu.dmean(src[a][f"{v}_clim_month"]), vu.dmean(src[b][f"{v}_clim_month"])
            d = 100 * (ma - mb) / mb if v == "pr" else ma - mb
            ax.plot(d.month, d, "o-", ms=3, color=col, label=nm)
        ax.axhline(0, color="grey", lw=0.5); ax.set_title(f"{lab}: {v} ({vu.dlabel(v)})", fontsize=8)
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "delta_annual_cycle.png", dpi=110); plt.show()

keys = ["pr_q_wet", "tasmax_q", "tasmin_q", "dtr_q", "rsds_q"]
fig, axes = plt.subplots(len(PAIRS), len(keys), figsize=(4.2 * len(keys), 3.1 * len(PAIRS)), squeeze=False)
for r, (a, b, lab) in enumerate(PAIRS):
    for c, k in enumerate(keys):
        ax = axes[r, c]
        if not all(k in s for s in (RAW[a], RAW[b], ADJ[a], ADJ[b])):
            ax.set_visible(False); continue
        rel = k.startswith("pr")
        for src, col, nm, mk in ((RAW, "k", "raw", "o-"), (ADJ, "tab:red", "F1", "s--")):
            d = (100 * (src[a][k] - src[b][k]) / src[b][k]) if rel else (src[a][k] - src[b][k])
            ax.plot(vu.QUANTILES, d.values, mk, ms=3, color=col, label=nm)
        ax.set_xscale("logit"); ax.axhline(0, color="grey", lw=0.5)
        ax.set_title(f"{lab}: {k.replace('_q_wet', ' wet').replace('_q', '')} ({'%' if rel else 'abs.'})", fontsize=8)
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "delta_quantiles.png", dpi=110); plt.show()

## Conclusions (to fill in)

- Calibration (historical + F1 vs ERA5): ...
- Climate-change and hosing deltas preserved? (slope, mean ratio; dry days): ...
- Points to discuss before the RCM bias adjustment: ...